# Utils

> Fill in a module description here

In [ ]:
#| default_exp utils

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export


In [ ]:
#| hide

# Enable I2C interface
sudo raspi-config
# Navigate to: Interface Options > I2C > Yes

# Reboot
sudo reboot

# Verify I2C is working
sudo i2cdetect -y 1
# You should see device at address 0x33 (default MLX90642 address)

In [ ]:
#| hide

# Update system

# Install Python packages
#sudo apt install python3-pip python3-dev python3-smbus i2c-tools -y
#pip3 install numpy matplotlib pillow

In [ ]:
import smbus
import time
import numpy as np

class MLX90642:
    def __init__(self, i2c_addr=0x33, i2c_bus=1):
        self.addr = i2c_addr
        self.bus = smbus.SMBus(i2c_bus)
        self.width = 32
        self.height = 24
        self.pixels = self.width * self.height
        
    def read_register(self, reg_addr):
        """Read 16-bit register from MLX90642"""
        try:
            # Read 2 bytes (16-bit register)
            data = self.bus.read_i2c_block_data(self.addr, reg_addr >> 8, 2)
            return (data[0] << 8) | data[1]
        except Exception as e:
            print(f"Error reading register {reg_addr:04X}: {e}")
            return 0
    
    def write_register(self, reg_addr, value):
        """Write 16-bit register to MLX90642"""
        try:
            data = [(reg_addr >> 8) & 0xFF, reg_addr & 0xFF, 
                   (value >> 8) & 0xFF, value & 0xFF]
            self.bus.write_i2c_block_data(self.addr, 0x3A, data)
            time.sleep(0.015)  # Wait for EEPROM write
        except Exception as e:
            print(f"Error writing register {reg_addr:04X}: {e}")
    
    def init_sensor(self):
        """Initialize the MLX90642 sensor"""
        print("Initializing MLX90642...")
        
        # Set measurement mode to continuous
        self.set_measurement_mode('continuous')
        
        # Set refresh rate to 4Hz
        self.set_refresh_rate(4)
        
        # Set output format to temperature
        self.set_output_format('temperature')
        
        print("MLX90642 initialized successfully!")
    
    def set_measurement_mode(self, mode):
        """Set measurement mode: 'continuous' or 'step'"""
        reg_val = self.read_register(0x11F4)
        if mode == 'continuous':
            reg_val &= ~0x0800
        else:  # step mode
            reg_val |= 0x0800
        self.write_register(0x11F4, reg_val)
    
    def set_refresh_rate(self, rate_hz):
        """Set refresh rate (2, 4, 8, 16 Hz)"""
        rate_map = {2: 2, 4: 3, 8: 4, 16: 5}
        if rate_hz not in rate_map:
            print("Invalid refresh rate. Using 4Hz.")
            rate_hz = 4
        
        reg_val = self.read_register(0x11F0)
        reg_val = (reg_val & ~0x0007) | rate_map[rate_hz]
        self.write_register(0x11F0, reg_val)
    
    def set_output_format(self, format_type):
        """Set output format: 'temperature' or 'raw'"""
        reg_val = self.read_register(0x11F4)
        if format_type == 'temperature':
            reg_val &= ~0x0100
        else:  # raw/normalized data
            reg_val |= 0x0100
        self.write_register(0x11F4, reg_val)
    
    def is_data_ready(self):
        """Check if new thermal data is available"""
        flags = self.read_register(0x3C14)
        return bool(flags & 0x0100)
    
    def get_thermal_image(self):
        """Get thermal image data"""
        # Wait for data to be ready
        timeout = 0
        while not self.is_data_ready() and timeout < 50:
            time.sleep(0.01)
            timeout += 1
        
        if timeout >= 50:
            print("Timeout waiting for thermal data")
            return None
        
        try:
            # Read thermal image data (768 pixels + 1 ambient temp)
            thermal_data = []
            for i in range(0, self.pixels + 1, 32):  # Read in chunks
                chunk_size = min(32, self.pixels + 1 - i)
                chunk = self.bus.read_i2c_block_data(self.addr, 0x34, chunk_size * 2)
                
                # Convert bytes to 16-bit values
                for j in range(0, len(chunk), 2):
                    if j + 1 < len(chunk):
                        value = (chunk[j] << 8) | chunk[j + 1]
                        # Convert to signed 16-bit
                        if value > 32767:
                            value -= 65536
                        thermal_data.append(value / 50.0)  # Temperature in °C
            
            # Reshape to 24x32 array (height x width)
            if len(thermal_data) >= self.pixels:
                return np.array(thermal_data[:self.pixels]).reshape(self.height, self.width)
            else:
                print(f"Insufficient data received: {len(thermal_data)} pixels")
                return None
                
        except Exception as e:
            print(f"Error reading thermal data: {e}")
            return None
    
    def get_ambient_temperature(self):
        """Get ambient temperature from sensor"""
        try:
            data = self.bus.read_i2c_block_data(self.addr, 0x3A, 2)
            temp_raw = (data[0] << 8) | data[1]
            if temp_raw > 32767:
                temp_raw -= 65536
            return temp_raw / 50.0  # Temperature in °C
        except Exception as e:
            print(f"Error reading ambient temperature: {e}")
            return 0.0